# Sentinel-2 seasonal composites via Google Earth Engine — parallel version

Downloads composites per year for chosen areas, using
**Sentinel-2 SR Harmonized**.

**TODO for LAURA**
1. Create a EART ENGINE project: https://console.cloud.google.com/projectcreate
2. Register the project: https://code.earthengine.google.com/register
3. Run the [authenticate cell](#Earth-Engine-Config) ONCE
4. Inititalize the project, remember to replace with your project name

## 1. Setup

In [28]:
import os
import re
import time
import logging
from io import BytesIO
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from dateutil.relativedelta import relativedelta


import ee
import requests
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from PIL import Image
from shapely import wkt

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

### Earth Engine Config

In [29]:
# ONLY NEEDED ONCE per machine.
#ee.Authenticate()

In [30]:
# CREATE A PROJECT IN THE EARTH ENGINE CONSOLE AND USE IT HERE, YOU NEED
# TO THEN REPLACE THE PROJECT NAME BELOW WITH YOUR OWN
ee.Initialize(project="landsat9-502716")

### Config

In [31]:
OUTPUT_DIR = "output"
NUTS_GEOJSON = "NUTS_RG_20M_2024_4326.geojson"
nuts_gdf = gpd.read_file(NUTS_GEOJSON)

DEFAULT_DIMENSIONS = 800      # max px on the long side; 1200 is ~40% faster
MAX_WORKERS = 8                # parallel EE requests; 4-8 is safe, higher risks 429s

RGB_BANDS = ["B4", "B3", "B2"]

## 2. Core Earth Engine functions

In [32]:
CS_PLUS_BAND = "cs_cdf"      # clear-sky probability, 0 (cloudy) → 1 (clear)
CLEAR_THRESHOLD = 0.60       # 0.5 = looser, 0.7 = stricter

def mask_clouds(image):
    """Cloud Score+ based masking — sharper than SCL, especially on thin clouds."""
    return image.updateMask(image.select(CS_PLUS_BAND).gte(CLEAR_THRESHOLD))


def apply_scale_factors(image, rgb_bands):
    """S2 SR: reflectance = DN / 10000 (no offset)."""
    optical = image.select(rgb_bands).divide(10000)
    return image.addBands(optical, None, True)


def safe_name(name):
    """Strip filesystem-hostile chars: slashes for Cataluña/Catalunya etc."""
    return re.sub(r'[/\\:*?"<>|]', " - ", name)

In [33]:
def get_composite_url(geometry, start_date, end_date,
                      dimensions=DEFAULT_DIMENSIONS, mask_water=False):
    """Build the composite and return a thumbnail URL. No .getInfo() round-trip —
    if the collection is empty, getThumbURL fails and we return None."""
    cs_plus = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")

    dataset = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
           .filterDate(start_date.strftime("%Y-%m-%d"), end_date.strftime("%Y-%m-%d"))
           .filterBounds(geometry)
           .linkCollection(cs_plus, [CS_PLUS_BAND]))    # attaches cs_cdf to each scene

    dataset = dataset.map(mask_clouds)
    dataset = dataset.map(lambda img: apply_scale_factors(img, RGB_BANDS))
    im = dataset.median()

    if mask_water:
        datamask = ee.Image("UMD/hansen/global_forest_change_2025_v1_13").select("datamask")
        im = im.updateMask(datamask.eq(1))

    im = im.updateMask(ee.Image.constant(1).clip(geometry).mask())

    try:
        return im.getThumbURL({
            "bands": RGB_BANDS,
            "min": 0.0,
            "max": 0.35,
            "region": geometry,
            "dimensions": dimensions,
            "format": "png",
            "crs": "EPSG:3857",
        })
    except ee.EEException as e:
        logging.error("Thumbnail URL failed: %s", e)
        return None

## 3. Download functions

The old per-region loop is now split into two things: a `build_jobs()` that lists every
(region, year, window) job, and a `run_jobs()` that runs them in a thread pool.

In [34]:
def download_image(url, subfolder, fname, timeout=300, retries=3):
    if url is None:
        return
    folder = os.path.join(OUTPUT_DIR, subfolder)
    os.makedirs(folder, exist_ok=True)

    for attempt in range(1, retries + 1):
        try:
            response = requests.get(url, timeout=timeout)
            if response.status_code != 200:
                logging.error("HTTP %s for %s: %s",
                              response.status_code, fname, response.text[:200])
                return
            Image.open(BytesIO(response.content)).save(
                os.path.join(folder, f"{fname}.png"))
            return
        except requests.exceptions.Timeout:
            logging.warning("Timeout for %s (attempt %s/%s)", fname, attempt, retries)
            time.sleep(5)
    logging.error("Gave up on %s after %s timeouts", fname, retries)


def process_job(job):
    pname, geometry, year, month, mask_water, dimensions = job

    folder_name = safe_name(pname)
    subfolder = os.path.join(folder_name, str(year))
    fname = f"{folder_name}_{year}-{month:02d}-01"

    if os.path.exists(os.path.join(OUTPUT_DIR, subfolder, f"{fname}.png")):
        return fname, 0, "skip"

    t0 = time.time()
    start_date = datetime(year, month, 1)
    end_date = start_date + relativedelta(months=1)

    url = get_composite_url(geometry, start_date, end_date,
                            dimensions=dimensions, mask_water=mask_water)
    t_url = time.time()

    if url is None:
        return fname, t_url - t0, "no-url"

    download_image(url, subfolder, fname)
    t_end = time.time()
    return fname, t_end - t0, f"ok (EE {t_url-t0:.0f}s, dl {t_end-t_url:.0f}s)"

In [35]:
def build_jobs(regions, start_year, end_year, months=None,
               mask_water=True, dimensions=DEFAULT_DIMENSIONS):
    """regions: list of (pname, geometry) pairs.
    months: list of month numbers (1-12). None or [] means all 12."""
    if not months:
        months = list(range(1, 13))

    jobs = []
    for pname, geometry in regions:
        for year in range(start_year, end_year + 1):
            for month in months:
                jobs.append((pname, geometry, year, month,
                             mask_water, dimensions))
    return jobs


def run_jobs(jobs, max_workers=MAX_WORKERS):
    """Run jobs in parallel. Prints progress and per-job timing."""
    total = len(jobs)
    logging.info("Running %s jobs with %s workers", total, max_workers)
    done = 0
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = [ex.submit(process_job, job) for job in jobs]
        for fut in as_completed(futures):
            done += 1
            try:
                fname, elapsed, status = fut.result()
                logging.info("[%d/%d] %s — %s", done, total, fname, status)
            except Exception as e:
                logging.error("[%d/%d] job crashed: %s", done, total, e)

In [36]:
EUROPE_BBOX = [-13, 34, 45, 72]   # [west, south, east, north]

def nuts_regions(level, countries=None, name_col="NUTS_NAME", id_col="NUTS_ID",
                 clip_bbox=EUROPE_BBOX):
    sel = nuts_gdf[nuts_gdf["LEVL_CODE"] == level]
    if countries is not None:
        sel = sel[sel["CNTR_CODE"].isin(countries)]

    clip = ee.Geometry.Rectangle(clip_bbox) if clip_bbox else None

    out = []
    for _, row in sel.iterrows():
        pname = f"{row[id_col]}_{row[name_col]}"
        geometry = ee.Geometry(row.geometry.__geo_interface__)
        if clip is not None:
            geometry = geometry.intersection(clip, ee.ErrorMargin(1))
        out.append((pname, geometry))
    return out

## 4. Usage examples

Use `build_jobs(regions, start_year, end_year, months)` to get satelitte images. If you leave months empty then every month of the year will be downloaded

In [ ]:
nuts_gdf_countries = nuts_gdf[nuts_gdf["LEVL_CODE"] == 0][["CNTR_CODE", "NUTS_NAME"]].drop_duplicates()
all_countries = list(nuts_gdf_countries.CNTR_CODE)

drought_countries = ["ES", "FR", "DE", "IT", "PT", "HU", "PL", "CZ", "LU", "BE", "CH", "NL", "AT"]

In [42]:
regions = nuts_regions(level=0, countries=all_countries)
jobs = build_jobs(regions, start_year=2026, end_year=2026, months = [7])
run_jobs(jobs)

INFO: Running 156 jobs with 8 workers
INFO: [1/156] SK_Slovensko_2025-07-01 — ok (EE 1s, dl 8s)
INFO: [2/156] SK_Slovensko_2024-07-01 — ok (EE 2s, dl 10s)
INFO: [3/156] SI_Slovenija_2022-07-01 — ok (EE 1s, dl 11s)
INFO: [4/156] SK_Slovensko_2023-07-01 — ok (EE 1s, dl 12s)
INFO: [5/156] SI_Slovenija_2024-07-01 — ok (EE 7s, dl 7s)
INFO: [6/156] SK_Slovensko_2022-07-01 — ok (EE 1s, dl 13s)
INFO: [7/156] SI_Slovenija_2025-07-01 — ok (EE 2s, dl 15s)
INFO: [8/156] SI_Slovenija_2023-07-01 — ok (EE 6s, dl 10s)
INFO: [9/156] TR_Türkiye_2024-07-01 — ok (EE 9s, dl 23s)
INFO: [10/156] XK_Kosovo - _2022-07-01 — ok (EE 1s, dl 6s)
INFO: [11/156] UA_Україна_2024-07-01 — ok (EE 3s, dl 34s)
INFO: [12/156] XK_Kosovo - _2023-07-01 — ok (EE 1s, dl 7s)
INFO: [13/156] TR_Türkiye_2022-07-01 — ok (EE 4s, dl 50s)
INFO: [14/156] XK_Kosovo - _2024-07-01 — ok (EE 1s, dl 10s)
INFO: [15/156] TR_Türkiye_2023-07-01 — ok (EE 5s, dl 52s)
INFO: [16/156] AL_Shqipëria_2023-07-01 — ok (EE 2s, dl 7s)
INFO: [17/156] XK_Kosovo

In [43]:
regions = nuts_regions(level=0, countries=all_countries)
jobs = build_jobs(regions, start_year=2026, end_year=2026, months = [7])
run_jobs(jobs)

INFO: Running 39 jobs with 8 workers
INFO: [1/39] XK_Kosovo - _2026-07-01 — ok (EE 1s, dl 4s)
INFO: [2/39] SI_Slovenija_2026-07-01 — ok (EE 2s, dl 8s)
INFO: [3/39] AL_Shqipëria_2026-07-01 — ok (EE 3s, dl 9s)
INFO: [4/39] AT_Österreich_2026-07-01 — ok (EE 5s, dl 7s)
INFO: [5/39] SK_Slovensko_2026-07-01 — ok (EE 3s, dl 10s)
ERROR: Thumbnail URL failed: Image.clip: The geometry for image clipping must not be empty.
INFO: [6/39] CY_Κύπρος _2026-07-01 — no-url
INFO: [7/39] BE_Belgique - België_2026-07-01 — ok (EE 1s, dl 7s)
INFO: [8/39] CH_Schweiz - Suisse - Svizzera_2026-07-01 — ok (EE 1s, dl 7s)
INFO: [9/39] BG_България _2026-07-01 — ok (EE 2s, dl 12s)
INFO: [10/39] BA_Bosna I Hercegovina_2026-07-01 — ok (EE 2s, dl 22s)
INFO: [11/39] DK_Danmark_2026-07-01 — ok (EE 2s, dl 13s)
INFO: [12/39] CZ_Česko_2026-07-01 — ok (EE 1s, dl 19s)
INFO: [13/39] EE_Eesti_2026-07-01 — ok (EE 3s, dl 12s)
INFO: [14/39] DE_Deutschland_2026-07-01 — ok (EE 10s, dl 24s)
INFO: [15/39] EL_Ελλάδα _2026-07-01 — ok (EE